In [1]:
import json
import pandas as pd
import os
import numpy as np
from tqdm import tqdm
os.chdir('..')

## Counterfact Data

In [2]:
path = 'hotel_dataset/counterfactsv3.5/corrected_splitopinion_typocorrected/indo_counterfacts.csv'
df_counterfact = pd.read_csv(path)
df_counterfact

,index,original_pair,corrupted_pair
0,4,"tidak dapat snack . setelah di keluhan , baru ...","baik sangat radio . setelah di keluhan , baru ..."
1,10,kamarnya oke . [A] [O] [S] [A] kamarnya [O] ok...,pot bunga busuk . [A] [O] [S] [A] pot bunga [...
2,18,tempat tdr kurang bersih . [A] [O] [S] [A] tem...,baju anakku cukup cepat . [A] [O] [S] [A] baju...
3,50,tetapi sayangnya di kamar yang saya tempati ti...,tetapi sayangnya di kamar yang saya tempati ce...
4,54,tidak ada sarapan . [A] [O] [S] [A] sarapan [O...,baik sangat lilin . [A] [O] [S] [A] lilin [O] ...
...,...,...,...
95,2399,"terimakasih airy , layanannya sangat sangat ba...","erimakasih airy , peraturan pahit, moga kedepa..."
96,2409,tidak ada lift . kesulitannya hanya mengangkut...,baik amat radio . kesulitannya hanya mengangku...
97,2415,kamar bersih hotel berada disamping gran mall ...,angin kotor hotel berada disamping gran mall b...
98,2458,karena terlalu banyak kamar jadi tidak diperha...,"tabil sekali , jadi proses berjalan sangat mul..."


In [3]:
from typing import List, Dict
import re
def parse_absa_string(text: str) -> List[Dict[str, str]]:
	"""
	Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
	Each dictionary contains the tag as the key and the corresponding value.
	For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
	[{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
	{'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

	Args:
		text (str): ABSA string output to be parsed.

	Returns:
		List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

	"""
	pattern = r"\[(\w+)\]\s*([^[]+)"
	matches = re.findall(pattern, text)

	result = []
	current_dict = {}

	for tag, content in matches:
		if tag == "SSEP":  # Sentence separator -> Start a new dictionary
			result.append(current_dict)
			current_dict = {}
		else:
			current_dict[tag] = content.strip()

	if current_dict:  # Append the last sentence if it exists
		result.append(current_dict)

	return result

def convert_to_gas_format(data_list, spaced=False):
	"""
	Converts a list of aspect-based sentiment dictionaries to the 
	extraction-style string format used in the GAS paper.

	Args:
		data_list: A list of dictionaries, where each dictionary must contain 
				the keys 'A' (Aspect), 'O' (Opinion), and 'S' (Sentiment).

	Returns:
		A single string formatted as '(A, O, S); (A, O, S); ...'
	"""
	# A list comprehension to create a formatted string for each dictionary
	# The order of elements in the tuple is specified as A, O, S
	if spaced:
		triplets = [f"( {item['A']} | {item['O']} | {item['S']} )" for item in data_list]
	else:
		triplets = [f"({item['A']}| {item['O']}| {item['S']})" for item in data_list]

	# Join the list of strings together with a semicolon and space
	if spaced:
		return " ; ".join(triplets)
	return "; ".join(triplets)

In [4]:
df_counterfact['original_pair_input'] = df_counterfact['original_pair'].apply(lambda x: x.split('[A] [O] [S]')[0].strip())
df_counterfact['original_pair_output'] = df_counterfact['original_pair'].apply(lambda x: convert_to_gas_format(parse_absa_string(x.split('[A] [O] [S]')[-1].strip()), spaced=True))
df_counterfact['original_pair_gas'] = df_counterfact.apply(lambda row: f"{row['original_pair_input']} => {row['original_pair_output']}", axis=1)

df_counterfact['corrupted_pair_input'] = df_counterfact['corrupted_pair'].apply(lambda x: x.split('[A] [O] [S]')[0].strip() if pd.notna(x) else x)
df_counterfact['corrupted_pair_output'] = df_counterfact['corrupted_pair'].apply(lambda x: convert_to_gas_format(parse_absa_string(x.split('[A] [O] [S]')[-1].strip()), spaced=True) if pd.notna(x) else x)
df_counterfact['corrupted_pair_gas'] = df_counterfact.apply(lambda row: f"{row['corrupted_pair_input']} => {row['corrupted_pair_output']}" if pd.notna(row['corrupted_pair_input']) and pd.notna(row['corrupted_pair_output']) else None, axis=1)

In [5]:
df_counterfact

,index,original_pair,corrupted_pair,original_pair_input,original_pair_output,original_pair_gas,corrupted_pair_input,corrupted_pair_output,corrupted_pair_gas
0,4,"tidak dapat snack . setelah di keluhan , baru ...","baik sangat radio . setelah di keluhan , baru ...","tidak dapat snack . setelah di keluhan , baru ...",( snack | tidak dapat | negative ),"tidak dapat snack . setelah di keluhan , baru ...","baik sangat radio . setelah di keluhan , baru ...",( radio | baik sangat | positive ),"baik sangat radio . setelah di keluhan , baru ..."
1,10,kamarnya oke . [A] [O] [S] [A] kamarnya [O] ok...,pot bunga busuk . [A] [O] [S] [A] pot bunga [...,kamarnya oke .,( kamarnya | oke | positive ),kamarnya oke . => ( kamarnya | oke | positive ),pot bunga busuk .,( pot bunga | busuk | negative ),pot bunga busuk . => ( pot bunga | busuk | ne...
2,18,tempat tdr kurang bersih . [A] [O] [S] [A] tem...,baju anakku cukup cepat . [A] [O] [S] [A] baju...,tempat tdr kurang bersih .,( tempat tidur | kurang bersih | negative ),tempat tdr kurang bersih . => ( tempat tidur |...,baju anakku cukup cepat .,( baju anakku | cukup cepat | positive ),baju anakku cukup cepat . => ( baju anakku | c...
3,50,tetapi sayangnya di kamar yang saya tempati ti...,tetapi sayangnya di kamar yang saya tempati ce...,tetapi sayangnya di kamar yang saya tempati ti...,( lampu tidur | tidak terdapat | negative ),tetapi sayangnya di kamar yang saya tempati ti...,tetapi sayangnya di kamar yang saya tempati ce...,( rekening tabungan | cepat sekali | positive ),tetapi sayangnya di kamar yang saya tempati ce...
4,54,tidak ada sarapan . [A] [O] [S] [A] sarapan [O...,baik sangat lilin . [A] [O] [S] [A] lilin [O] ...,tidak ada sarapan .,( sarapan | tidak ada | negative ),tidak ada sarapan . => ( sarapan | tidak ada |...,baik sangat lilin .,( lilin | baik sangat | positive ),baik sangat lilin . => ( lilin | baik sangat |...
...,...,...,...,...,...,...,...,...,...
95,2399,"terimakasih airy , layanannya sangat sangat ba...","erimakasih airy , peraturan pahit, moga kedepa...","terimakasih airy , layanannya sangat sangat ba...",( layanannya | sangat sangat baik | positive ),"terimakasih airy , layanannya sangat sangat ba...","erimakasih airy , peraturan pahit, moga kedepa...",( peraturan | pahit | negative ),"erimakasih airy , peraturan pahit, moga kedepa..."
96,2409,tidak ada lift . kesulitannya hanya mengangkut...,baik amat radio . kesulitannya hanya mengangku...,tidak ada lift . kesulitannya hanya mengangkut...,( lift | tidak ada | negative ),tidak ada lift . kesulitannya hanya mengangkut...,baik amat radio . kesulitannya hanya mengangku...,( radio | baik amat | positive ),baik amat radio . kesulitannya hanya mengangku...
97,2415,kamar bersih hotel berada disamping gran mall ...,angin kotor hotel berada disamping gran mall b...,kamar bersih hotel berada disamping gran mall ...,( kamar | bersih | positive ),kamar bersih hotel berada disamping gran mall ...,angin kotor hotel berada disamping gran mall b...,( angin | kotor | negative ),angin kotor hotel berada disamping gran mall b...
98,2458,karena terlalu banyak kamar jadi tidak diperha...,"tabil sekali , jadi proses berjalan sangat mul...",karena terlalu banyak kamar jadi tidak diperha...,( kebersihan | karena terlalu banyak kamar jad...,karena terlalu banyak kamar jadi tidak diperha...,"tabil sekali , jadi proses berjalan sangat mul...","( ekonomi | stabil sekali , jadi proses berjal...","tabil sekali , jadi proses berjalan sangat mul..."


In [6]:
df_counterfact[['index', 'original_pair_gas', 'corrupted_pair_gas']].rename({'original_pair_gas': 'original_pair', 'corrupted_pair_gas': 'corrupted_pair'}).to_csv('utils_notebooks/gas_counterfact_converted.csv', index=False)

## Whole dataset

In [3]:
dataset_folder = 'corrected_splitopinion_typocorrected_gas_arrow'
split = 'train'
lang = 'indo'
dataset_per_split = {}
for split in ['train', 'test']:
	dataset_path = f'hotel_dataset/{lang}/{dataset_folder}/hotel_aste_{split}_augmented{"_noreasoning" if split == "train" else ""}.json'
	with open(dataset_path, 'r') as f:
		data = json.load(f)
	dataset_per_split[split] = data

In [12]:
import re
def add_space_around_punctuation(text):
    # Except for '-'
    # Ensure space before punctuation
    text = re.sub(r'(\S)([.,!?\(\)\"\';:+/]+)', r'\1 \2', text)
    # Ensure space after punctuation
    text = re.sub(r'([.,!?\(\)\"\';:+/]+)(\S)', r'\1 \2', text)
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    # Ensure punctuation sequences like '...' are split into spaced dots
    text = re.sub(r'([.]{2,})', lambda m: ' '.join(m.group(1)), text)
    return text.strip()

In [13]:
add_space_around_punctuation(dataset_per_split['train'][0]['target'])

'( ac , tidak berfungsi optimal , negative ) ; ( wifi koneksi , kurang stabil , negative )'

In [10]:
new_dataset_per_split = {}
for split in ['train', 'test']:
	new_data = []
	for item in dataset_per_split[split]:
		new_item = item.copy()
		new_item['target'] = add_space_around_punctuation(item['target'])
		new_data.append(new_item)
	new_dataset_per_split[split] = new_data

In [11]:
# Store the new dataset per split
for split in ['train', 'test']:
	output_path = f'hotel_dataset/{lang}/{dataset_folder}_spaced/hotel_aste_{split}_augmented{"_noreasoning" if split == "train" else ""}_gas.json'
	os.makedirs(os.path.dirname(output_path), exist_ok=True)
	with open(output_path, 'w') as f:
		json.dump(new_dataset_per_split[split], f, indent=4)

## GAS with |

In [11]:
dataset_folder = 'corrected_splitopinion_typocorrected_aos'
split = 'train'
lang = 'indo'
dataset_per_split = {}
for split in ['train', 'test']:
	dataset_path = f'hotel_dataset/{lang}/{dataset_folder}/hotel_aste_{split}_augmented{"_noreasoning" if split == "train" else ""}.json'
	with open(dataset_path, 'r') as f:
		data = json.load(f)
	dataset_per_split[split] = data

In [13]:
new_dataset_per_split = {}
for split in ['train', 'test']:
	new_data = []
	for item in dataset_per_split[split]:
		new_item = item.copy()
		new_item['input'] = new_item['input'].replace('[A] [O] [S]', '').strip() + ' =>'
		new_item['target'] = convert_to_gas_format(parse_absa_string(item['target']), spaced=True)
		new_data.append(new_item)
	new_dataset_per_split[split] = new_data

In [14]:
new_dataset_per_split

{'train': [{'sentence_id': 0,
   'instance_id': 0,
   'task_elements': 'aos',
   'input': 'kamar saya ada kendala di ac tidak berfungsi optimal . dan juga wifi koneksi kurang stabil . =>',
   'target': '( ac | tidak berfungsi optimal | negative ) ; ( wifi koneksi | kurang stabil | negative )',
   'element_order': 'aos'},
  {'sentence_id': 1,
   'instance_id': 5,
   'task_elements': 'aos',
   'input': 'tempatnya bagus . kolam renangnya bersih . =>',
   'target': '( tempatnya | bagus | positive ) ; ( kolam renangnya | bersih | positive )',
   'element_order': 'aos'},
  {'sentence_id': 2,
   'instance_id': 10,
   'task_elements': 'aos',
   'input': 'oke banget , tetapi ac nya tidak bisa diatur suhu nya . =>',
   'target': '( ac nya | tidak bisa diatur suhu nya | negative ) ; ( null | oke banget | positive )',
   'element_order': 'aos'},
  {'sentence_id': 3,
   'instance_id': 15,
   'task_elements': 'aos',
   'input': 'keren . nyaman semuanya . =>',
   'target': '( semuanya | nyaman | posi

In [15]:
# Save the new dataset per split
for split in ['train', 'test']:
	output_path = f'hotel_dataset/{lang}/corrected_splitopinion_typocorrected_gas_arrow_bar/hotel_aste_{split}_augmented{"_noreasoning" if split == "train" else ""}_gas.json'
	os.makedirs(os.path.dirname(output_path), exist_ok=True)
	with open(output_path, 'w') as f:
		json.dump(new_dataset_per_split[split], f, indent=4)